# 03｜分类与边界框回归：一条检测结果怎样产生

前两课已经回答了两个问题：

1. 目标检测模型要输出类别和边界框。
2. IoU 可以衡量预测框与真实框的重叠程度。

现在继续追问：同一个神经网络怎样既判断“这是什么”，又预测“它在哪里”？

这一课只讲分类与边界框回归怎样组成一条检测结果。暂时不讲模型怎样决定哪个预测负责哪个真实物体。

## 1. 一条检测结果包含两种性质不同的答案

假设图片中有一只狗，它的检测结果可以写成：

$$
(\text{狗},c_x,c_y,w,h)
$$

这里包含两种不同类型的问题：

- 类别“狗”是一个离散选项。
- 边界框坐标是四个连续数值。

因此，检测模型不能只使用一个普通分类输出层。它需要从同一份视觉特征中产生两个分支：

$$
\text{视觉特征}\rightarrow\begin{cases}\text{分类分支：预测是什么}\\\text{边界框分支：预测在哪里}\end{cases}
$$

## 2. 什么是分类问题

分类问题从有限个类别中选择答案。假设数据集只有猫、狗和汽车三个物体类别，分类分支需要判断当前预测更像哪一类。

它通常先输出一组未经归一化的类别分数：

$$
z_{cls}=(z_{cat},z_{dog},z_{car})
$$

这些分数称为 logits。经过 Softmax 或 Sigmoid 等变换后，才能得到更容易解释的类别分数或概率。

分类分支关心的是：

> 当前这条预测应该属于哪个离散类别？

## 3. 什么是回归问题

回归问题预测连续数值。边界框可以写成：

$$
b=(c_x,c_y,w,h)
$$

也可以写成：

$$
b=(x_{min},y_{min},x_{max},y_{max})
$$

这些坐标不是从“左、中、右”几个固定标签中选择，而是在一个连续范围里预测具体数值，所以称为边界框回归。

这里的“回归”和以前学习线性回归时的核心含义相同：输出是连续数值。但检测中的输入特征、输出数量和损失设计会复杂得多。

## 4. 分类与回归为什么不能互相替代

如果只做分类，模型可以说“这里是一只狗”，却无法给出狗的位置。

如果只做边界框回归，模型可以给出一个矩形，却无法说明框里是狗、猫还是汽车，也无法判断这里是否根本没有物体。

所以一条完整检测结果必须同时满足：

$$
\text{有效检测}=\text{类别正确}+\text{位置足够准确}
$$

分类和回归不是先后替代关系，而是同一条检测结果的两个组成部分。

## 5. 什么是预测槽位

神经网络输出必须是规则张量，但不同图片中的真实物体数量不同。为方便讨论，我们先使用一个与具体模型无关的概念：**预测槽位**。

每个预测槽位都尝试输出一条检测结果：

$$
\text{第 }i\text{ 个槽位}ightarrow(\text{类别预测}_i,\text{边界框预测}_i)
$$

预测槽位在不同模型中的来源不同：

- YOLO 类模型会在多个空间位置和尺度上产生密集预测。
- DETR 使用固定数量的 Object Queries 产生预测。

现在不研究槽位怎样产生，只关心每个槽位内部要输出什么。

## 6. 共享特征为什么可以接两个预测头

设某个预测槽位已经得到一个 $D$ 维特征向量：

$$
f_i\in\mathbb{R}^{D}
$$

这个向量包含模型为当前槽位收集到的视觉信息。它可以同时送入两个不同的预测头：

$$
\begin{aligned}
z_i^{cls}&=g_{cls}(f_i)\\
\hat{b}_i&=g_{box}(f_i)
\end{aligned}
$$

其中：

- $g_{cls}$：分类头，把特征映射为类别 logits。
- $g_{box}$：边界框回归头，把特征映射为 4 个坐标。

两个头输入相同，但参数和任务不同。分类头学习类别边界，回归头学习空间位置。

## 7. 一批预测的张量形状

设：

- batch 大小为 $B$。
- 每张图片有 $N$ 个预测槽位。
- 每个槽位的特征维度为 $D$。
- 分类输出维度为 $K_{cls}$。

共享特征形状为：

$$
F\in\mathbb{R}^{B\times N\times D}
$$

经过两个预测头后：

$$
\begin{aligned}
Z_{cls}&\in\mathbb{R}^{B\times N\times K_{cls}}\\
\hat{B}&\in\mathbb{R}^{B\times N\times4}
\end{aligned}
$$

最后一维的 4 就是一组边界框坐标。分类维度则取决于模型如何表示物体类别与背景。

## 8. “没有物体”怎样表示

预测槽位数量通常多于真实物体数量，因此许多槽位必须能够表达“这里没有需要输出的物体”。常见设计有两类。

### 方式一：增加背景或 `no object` 类

如果有 $K$ 个真实物体类别，分类头输出 $K+1$ 个 logits，最后一个类别表示没有物体。原始 DETR 使用这种思路。

### 方式二：单独预测目标性分数

模型额外判断当前槽位是否包含物体，再预测类别。许多 YOLO 类模型采用过目标性或相近设计，但不同版本细节并不完全相同。

两种方式的共同目标是：允许一部分预测槽位成为有效物体，另一部分槽位表示背景。

## 9. 用四个预测槽位建立直觉

假设数据集只有猫、狗和汽车三个类别，模型为一张图片准备了 4 个预测槽位。前向传播后可能得到：

| 槽位 | 分类结果 | 预测框 | 当前直观含义 |
|---:|---|---|---|
| 1 | 狗 | $(0.30,0.45,0.20,0.35)$ | 可能是一只狗 |
| 2 | 汽车 | $(0.65,0.55,0.40,0.25)$ | 可能是一辆汽车 |
| 3 | 没有物体 | $(0.10,0.20,0.30,0.30)$ | 这个框不会作为有效结果 |
| 4 | 狗 | $(0.32,0.46,0.22,0.36)$ | 可能与槽位 1 重复 |

这张表暂时暴露出三个问题：

1. 哪些槽位应该学习真实目标？
2. 槽位 1 和槽位 4 是否重复检测同一只狗？
3. 没有物体的槽位是否需要学习它输出的边界框？

这些问题会逐步引出目标分配、正负样本与 NMS。

## 10. 分类损失负责纠正什么

分类损失比较预测类别与目标类别。它需要推动：

- 负责真实物体的槽位提高正确类别分数。
- 不负责物体的槽位提高背景或 `no object` 分数。
- 错误类别的分数降低。

常见分类损失包括交叉熵、二元交叉熵和 Focal Loss。选择哪一种取决于类别表示、正负样本比例和模型设计。

现在不需要把某种损失固定绑定到所有检测器。先记住共同作用：

> 分类损失告诉模型每个预测槽位应该回答“是什么”还是“没有物体”。

## 11. 边界框损失负责纠正什么

对于负责真实物体的槽位，边界框损失比较预测框 $\hat{b}_i$ 与真实框 $b_i$。

最直接的方法是比较四个坐标的差异。例如 L1 损失：

$$
\mathcal{L}_{L1}=\lVert\hat{b}_i-b_i\rVert_1
$$

它等于四个坐标绝对误差之和。

检测器还常使用基于 IoU 的损失，从框的整体几何关系衡量定位质量。不同模型可能组合 L1、Smooth L1、GIoU、DIoU 或 CIoU 等方法。

边界框损失的共同目标是：让预测框逐渐靠近负责的真实框。

## 12. 为什么背景槽位通常不计算边界框回归损失

如果某个槽位被判断为背景或没有物体，就不存在与它对应的真实边界框。

假设强行要求所有背景槽位都回归到全零框，模型就会学习一个与检测任务无关的虚假目标，而且大量背景槽位可能压过真正物体的定位梯度。

因此，可以用一个正样本指示量 $m_i$ 表示是否计算第 $i$ 个槽位的框损失：

$$
m_i=\begin{cases}1,&\text{槽位负责一个真实物体}\\0,&\text{槽位为背景或没有物体}\end{cases}
$$

整体框损失可以概念性地写成：

$$
\mathcal{L}_{box}=\frac{\sum_i m_i\,\ell_{box}(\hat{b}_i,b_i)}{\max(1,\sum_i m_i)}
$$

只有 $m_i=1$ 的槽位真正贡献边界框回归误差。

## 13. 两类损失怎样组成总损失

分类与边界框回归必须一起训练，因此总损失通常是多个部分的加权和：

$$
\mathcal{L}_{total}=\lambda_{cls}\mathcal{L}_{cls}+\lambda_{box}\mathcal{L}_{box}
$$

如果框损失还包含坐标损失与 IoU 类损失，可以继续拆开：

$$
\mathcal{L}_{total}=\lambda_{cls}\mathcal{L}_{cls}+\lambda_{coord}\mathcal{L}_{coord}+\lambda_{iou}\mathcal{L}_{iou}
$$

系数 $\lambda$ 用来平衡不同损失的数值尺度和任务重要程度。它们不是学习率，也不是模型预测出来的类别概率。

## 14. 两种损失怎样共同训练共享特征

前向传播时，同一个槽位特征分别进入分类头和边界框头。反向传播时，两种损失都会产生梯度：

$$
\begin{aligned}
\mathcal{L}_{cls}&\rightarrow g_{cls}\rightarrow\text{共享特征}\\
\mathcal{L}_{box}&\rightarrow g_{box}\rightarrow\text{共享特征}
\end{aligned}
$$

分类梯度推动共享特征更容易区分类别，框回归梯度推动共享特征保留位置、边缘和尺度信息。

所以检测模型学到的特征必须同时服务于语义和空间：既要知道“是什么”，又不能丢失“在哪里”。

## 15. 为什么不能先随便计算损失

假设图片中有一只狗，但模型产生了许多预测槽位。计算损失前必须先回答：

- 哪个槽位负责这只狗？
- 哪些槽位应该被当成背景？
- 如果有多个真实物体，每个槽位应该与哪一个真实框比较？

如果没有确定对应关系，就无法给分类头准备正确标签，也无法知道预测框应该与哪个真实框计算回归损失。

因此，训练中的逻辑顺序是：

$$
\text{模型产生预测}ightarrow\text{确定预测与真实目标的对应关系}ightarrow\text{计算分类和框损失}
$$

中间这一步就是下一课要学习的目标分配。

## 16. YOLO 与 DETR 在这里有什么共同点

YOLO 和 DETR 组织预测槽位的方法不同，但最终都要回答类别与位置。

| 角度 | YOLO 类模型 | DETR |
|---|---|---|
| 预测槽位来源 | 特征图上的密集位置与尺度 | Object Queries |
| 类别输出 | 类别分数，常结合目标性设计 | $K+1$ 类，包含 `no object` |
| 框输出 | 常预测相对网格或参考框的参数 | 常直接预测归一化 `cxcywh` |
| 共同任务 | 判断是什么 | 判断是什么 |
| 共同任务 | 预测在哪里 | 预测在哪里 |

现在先抓住共同骨架。具体差异要等学习完目标分配和重复预测以后再比较。

## 17. 常见误区

### 误区一：边界框回归就是再做一次分类

分类从有限标签中选择；回归预测连续坐标，输出空间不同。

### 误区二：分类正确就代表检测正确

类别正确但框严重错位，仍然不是合格检测。

### 误区三：背景槽位的边界框应该回归到 0

背景没有对应真实框，其框输出通常不参与回归损失。

### 误区四：损失权重就是学习率

损失权重平衡不同任务；学习率控制优化器每次更新参数的步长。

### 误区五：所有检测模型都用完全相同的分类头和框损失

共同任务相同，但类别表示、框参数化和损失选择可以不同。

## 18. 本节小结

这一课需要真正记住七个结论：

1. 一条检测结果同时包含离散类别和连续边界框坐标。
2. 分类头回答“是什么”，边界框回归头回答“在哪里”。
3. 两个预测头可以读取同一份共享视觉特征，但拥有不同参数。
4. 每个预测槽位都会产生类别输出和 4 个边界框坐标。
5. 背景或 `no object` 槽位参与分类学习，但通常不参与边界框回归。
6. 总损失由分类损失与一种或多种边界框损失加权组成。
7. 计算损失前，必须先确定预测槽位与真实目标的对应关系。

完整逻辑是：

$$
\text{共享特征}\rightarrow\begin{cases}\text{分类头}\rightarrow\text{类别}\\\text{框回归头}\rightarrow\text{边界框}\end{cases}\rightarrow\text{一条检测结果}
$$

下一课将慢慢解释正样本、负样本和目标分配：模型究竟怎样决定哪个预测槽位负责哪个真实物体。

## 19. 自测问题

1. 为什么一条检测结果既包含分类问题，又包含回归问题？
2. 离散类别与连续边界框坐标的区别是什么？
3. 边界框回归与以前学习的回归有什么共同点？
4. 什么是预测槽位？
5. YOLO 与 DETR 的预测槽位分别来自哪里？
6. 设共享特征形状为 $B\times N\times D$，边界框头输出什么形状？
7. 分类维度为什么有时是 $K+1$？
8. `no object` 与目标性分数分别怎样表达背景？
9. 为什么背景槽位通常不计算边界框损失？
10. 分类损失主要纠正什么？
11. 边界框损失主要纠正什么？
12. 为什么总损失需要不同的权重系数？
13. 分类与回归梯度是否会影响共享特征？
14. 为什么必须先做目标分配，再计算损失？

### 自测参考答案

1. 模型既要从有限类别中判断物体是什么，又要预测表示位置和大小的连续坐标。
2. 类别从有限标签中选择，边界框坐标可以在连续范围内取值。
3. 两者都根据输入特征预测连续数值。
4. 它是模型预先提供的一条候选输出位置，每个槽位尝试产生一个类别和一个边界框。
5. YOLO 类模型通常来自特征图上的密集位置和尺度；DETR 来自 Object Queries。
6. $B\times N\times4$。
7. 除了 $K$ 个真实物体类别，还增加一个背景或 `no object` 类。
8. 前者把背景作为额外类别；后者用单独分数判断当前位置是否有目标。
9. 背景没有对应的真实边界框，强行回归会制造虚假目标。
10. 提高正确类别或背景的分数，降低错误类别分数。
11. 让预测框在位置、大小和重叠质量上靠近真实框。
12. 不同损失的数值尺度和任务作用不同，需要进行平衡。
13. 会，两种梯度都会通过各自预测头流回共享特征。
14. 不先确定对应关系，就不知道每个槽位的类别标签，也不知道预测框该与哪个真实框比较。